In [1]:
import aspose.words_foss as aw 

docs_path = './docs/'
filename = "uk-rf.doc"

doc = aw.Document(docs_path+filename)
text = doc.get_text()

In [2]:
import numpy as np

def split_into_chunks(text: str, chunk_size: int,  overlap: int=0) -> list[str]:
    # для каждого чанка выведем начальный и конечный индексы, чтобы 
    if overlap >= chunk_size:
        raise ValueError("chunk_size должен быть больше overlap!")

    if overlap < 0:
        raise ValueError("overlap не должен быть меньше нуля!")

    if chunk_size < 0:
        raise ValueError("chunk_size не может быть меньше нуля!")

    if not text:
        raise ValueError("Текст пустой!")

    start_indexes = np.arange(0, len(text), chunk_size-overlap)
    end_indexes = start_indexes + chunk_size

    chunks = [
        text[start:end] for start, end in zip(start_indexes, end_indexes)
    ]

    return chunks

In [ ]:
import torch
import torch.nn.functional as F
import polars as pl
import hashlib
from transformers import AutoTokenizer, AutoModel

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

cache_dir = './models/'
embeding_model_name = "google/embeddinggemma-2"

vdb_path = './database/'
vdb_filename = 'test.parquet'

tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

def get_ids(chunks, tokenizer):
    tokens = [tokenizer.tokenize(chunk) for chunk in chunks]

    ids = [tokenizer.convert_tokens_to_ids(token) for token in tokens]

    prepared_ids = tokenizer.pad(
        {"input_ids": ids},
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return prepared_ids

def get_embeddings(batch, model):
    with torch.inference_mode():
        output = model(**batch)

    token_embeddings = output.last_hidden_state
    token_embeddings_prepared = (
        batch['attention_mask']
        .to(device)
        .to(torch.float64)
        .unsqueeze(-1)
    ) * (
        token_embeddings
    )

    return token_embeddings_prepared.mean(dim=1).cpu()

overlap = 40
chunk_size = 512

chunks = split_into_chunks(text, chunk_size, overlap)

ids = tokenizer(
    chunks,
    return_tensors="pt",
    padding=True
)

batch_size = 128
tensor_len = ids['input_ids'].shape[0]

embeddings = []

for i in np.arange(0, tensor_len, batch_size):
    batch = {
        key: value[i:i+batch_size].to(device)
        for key, value in ids.items()
    }

    with torch.inference_mode():
        output = F.normalize(
            model(**batch).last_hidden_state,
            dim=1
        )
        output = output.mean(dim=1)
    
    embeddings.append(output.to("cpu"))

embeddings = torch.concat(embeddings, dim=0)
del output; del model; del tokenizer

table = pl.DataFrame({
    "id": [hashlib.sha256(chunk.encode('utf-8')).hexdigest() for chunk in chunks],
    "text": chunks,
    "embedding_vector": embeddings.to(torch.float64).numpy(),
    "doc": filename
})

table.write_parquet(
    vdb_path+vdb_filename
)

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [5]:
# search in base
query1 = "Мне нужна узнать: могу ли я своровать и быть безнаказанным"
query2 = "В какой статье есть наказание за убийство?"
query3 = "Какой кодекс в арбитражном суде?"

def search(query, model, tokenizer, base_file):
    # tokenizer

    # embedding_vector = get_embeddings()
    pass

In [43]:
tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

tokens = tokenizer(
    query1, 
    attention_mask=True,
    return_tensors="pt"
).to(device)

output = F.normalize(model(**tokens).last_hidden_state, dim=1)
output = output.mean(dim=1).to(torch.float64).cpu()

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [58]:
def cosine_similarity(x, y):
    return x @ y

db = pl.read_parquet(vdb_path+vdb_filename)
vector_db = torch.tensor(db['embedding_vector'].to_list(), dtype=torch.float64)
top_k_scores, top_k_indices = torch.topk(vector_db @ output.squeeze(0), k=5)

result = (
    db[top_k_indices.cpu().numpy()]
)

In [ ]:
def retrieval(batches, query_embeddings):
    for batch in batches:
        score = cosine_similarity(batch, query_embeddings)